In [8]:
import pandas as pd

search_df = pd.read_parquet("../../data/raw_data/search_dataset.parquet")

In [9]:
### utils ###
import regex as re

def detect_script(text):
    if pd.isna(text):
        return 'unknown'
    
    text = str(text)
    has_arabic = bool(re.search(r'\p{Arabic}', text))
    has_latin = bool(re.search(r'\p{Latin}', text))
    
    if has_arabic and has_latin:
        return 'mixed'
    elif has_arabic:
        return 'arabic'
    elif has_latin:
        return 'latin'
    else:
        return 'other'


def match_substring_vec(col_a, col_b):
    a = col_a.fillna("").str.lower()
    b = col_b.fillna("").str.lower()
    return pd.Series(
        [kw in prod if kw != "" else False for kw, prod in zip(a, b)],
        index=col_a.index
    ).astype(int)

## Data Preparation
 
### Unit of observation
 
Each row in the dataset is a **product impression**: one product shown to one user for one search keyword at one position. The dataset contains ~3.9M impressions after filtering rows with null `designer` or `keyword` (~5% of data).
 
### Session construction
 
The dataset has no native session identifier. A single `user_id × keyword` pair can appear multiple times if the user repeated the same search. To isolate individual search events, sessions were constructed using the **index-reset method**: within each `user_id × keyword` group (sorted by timestamp), a monotonically increasing sequence of `index` values belongs to one session. When the index drops, a new session begins. This avoids arbitrary time-window thresholds and is grounded in how the search system serves results.
 
The session signature is the composite key `(user_id, keyword, query_id)`, yielding **255,248 unique query sessions**.
 
### Granularity contract
 
- The **model** operates at the impression level. It scores each product independently
- **Evaluation** operates at the session level. Impressions are grouped into sessions, ranked by predicted score, and MRR is computed per session then averaged
- The **train/test split** operates at the session level. All impressions from one session stay in the same split (80/20) 

In [10]:
search_df_model = search_df[
    (search_df["designer"].notna()) & 
    (search_df["keyword"].notna())
].copy()

In [11]:
#  user_id and keyword aren't enough to identify a query session
#  we utilize the index for a particular user_id x keyword pair to identify a cutoff point
#  that point allows us to use a query_id label that we can group by later on to identify and analyze unique query sessions
#  noting that a user_id x keyword pair can occur multiple times across different sessions (assumption: a user can run the same query multiple times) 
#  role of using ts here is to enable the split logic. For example, a user can query "Dior" at 10 AM and then later at 2 PM. The results should not be mixed, hence why we include it in the group by.
#  
search_df_model = search_df_model.sort_values(["user_id", "keyword", "ts", "index"])
search_df_model["index_reset"] = search_df_model.groupby(["user_id", "keyword"])["index"].diff() <= 0
search_df_model["index_reset"] = search_df_model["index_reset"].fillna(True)  # first row of each group starts a session
search_df_model["query_id"] = search_df_model.groupby(["user_id", "keyword"])["index_reset"].cumsum()

"""
In other words: within each user-keyword pair, monotonically increasing index values belong to the same search event, 
and any index that drops signals a new one.  The query_id counter labels which search event each row belongs to.
"""

query_signature_final_cols = ["user_id", "keyword", "query_id"]

## Feature Engineering
 
Features are organized into four groups based on what they capture:
 
### Query features
- `keyword_length`: character length of the search keyword
- `tfidf_similarity`: cosine similarity between TF-IDF vectors of the keyword and product name, fitted on the combined vocabulary of all keywords and product names (max 5,000 features). This is the only continuous query-product relevance signal in the feature set.
 
### Product features (aggregate CTR)
- `product_ctr`: historical click-through rate at the product level
- `designer_ctr`: CTR aggregated at the designer level
- `productclass_ctr`: CTR at the product class level
- `class_ctr`: CTR at the class level
 
These are **query-independent**: they reflect global popularity, not relevance to a specific query. As the ablation study shows (below), these features hurt ranking performance.
 
### Interaction features (query-product matching)
- `kw_matches_designer`: 1 if the keyword contains a known designer name
- `kw_matches_class`: 1 if the keyword is a substring of the product's class
- `kw_matches_productclass`: 1 if the keyword is a substring of the product class
- `kw_in_product_name`: 1 if the keyword is a substring of the product name
 
These are binary and use exact substring matching, which misses morphological variants ("dresses" vs "dress") and cross-lingual matches (Arabic keywords vs English product names). Use of lemmetization or stemming can potentially improve coverage here (decided not to pursue that to limit complexity of this run).
 
### Categorical features (one-hot encoded)
- `keyword_lang`: script detection (Latin, Arabic, mixed, other)
- `intent`: navigational (keyword matches a known designer) vs exploratory
 
### Position feature
- `index`: the position assigned by the current ranking system. Included during training so the model can learn to separate position effect from relevance. **Neutralized at inference** by setting index to a constant (training mean) for all rows, so ranking is based purely on predicted relevance.


In [70]:
query_features = ["keyword_length","tfidf_similarity",]
product_features = ["product_ctr", "designer_ctr", "productclass_ctr", "class_ctr"]
interaction_features = ["kw_matches_designer", "kw_matches_class", "kw_matches_productclass", "kw_in_product_name"]
one_hot_features = ["lang_latin", "lang_mixed", "lang_other", "intent_navigational"]


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
import numpy as np

# Fit on all product names + keywords combined
corpus = pd.concat([search_df_model["name"].fillna(""), search_df_model["keyword"].fillna("")]).unique()
tfidf = TfidfVectorizer(max_features=5000)
tfidf.fit(corpus)

# Transform and compute row-wise similarity
kw_vectors = tfidf.transform(search_df_model["keyword"].fillna(""))
name_vectors = tfidf.transform(search_df_model["name"].fillna(""))

# Row-wise cosine similarity (not full pairwise matrix)
search_df_model["tfidf_similarity"] = 1 - paired_cosine_distances(kw_vectors, name_vectors)


In [14]:
search_df_model['keyword_lang'] = search_df_model['keyword'].apply(detect_script)
search_df_model['keyword_length'] = search_df_model['keyword'].str.len()
known_designers = set(search_df_model["designer"].dropna().str.lower().unique())
search_df_model["intent"] = search_df_model["keyword"].str.lower().apply(
    lambda x: "navigational" if any(d in x for d in known_designers) else "exploratory"
)

In [15]:
product_stats = search_df_model.groupby("product_id").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
product_stats["product_ctr"] = product_stats["clicks"] / product_stats["views"]

designer_stats = search_df_model.groupby("designer").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
designer_stats["designer_ctr"] = designer_stats["clicks"] / designer_stats["views"]

product_class_stats = search_df_model.groupby("productclass").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
product_class_stats["productclass_ctr"] = product_class_stats["clicks"] / product_class_stats["views"]

class_stats = search_df_model.groupby("class").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
class_stats["class_ctr"] = class_stats["clicks"] / class_stats["views"]

search_df_model = search_df_model.merge(
    product_stats[["product_ctr"]].reset_index(), 
    on="product_id", 
    how="left"
)

search_df_model = search_df_model.merge(
    designer_stats[["designer_ctr"]].reset_index(), 
    on="designer", 
    how="left"
)

search_df_model = search_df_model.merge(
    product_class_stats[["productclass_ctr"]].reset_index(), 
    on="productclass", 
    how="left"
)

search_df_model = search_df_model.merge(
    class_stats[["class_ctr"]].reset_index(), 
    on="class", 
    how="left"
)

In [36]:
kw = search_df_model["keyword"].str.lower()

search_df_model["kw_matches_designer"] = search_df_model["keyword"].str.lower().apply(
    lambda x: 1 if any(d in x for d in known_designers) else 0
)

search_df_model["kw_matches_class"] = match_substring_vec(kw, search_df_model["class"])
search_df_model["kw_matches_productclass"] = match_substring_vec(kw, search_df_model["productclass"])
search_df_model["kw_in_product_name"] = match_substring_vec(kw, search_df_model["name"])

In [16]:
lang_dummies = pd.get_dummies(search_df_model["keyword_lang"], prefix="lang", drop_first=True).astype(int)
intent_dummies = pd.get_dummies(search_df_model["intent"], prefix="intent", drop_first=True).astype(int)
search_df_model = pd.concat([search_df_model, lang_dummies, intent_dummies], axis=1)

In [34]:
search_df_model.columns

Index(['dt', 'ts', 'user_id', 'app_country', 'keyword', 'index', 'product_id',
       'productclass', 'class', 'designer', 'name', 'plp_view', 'plp_click',
       'is_converted', 'index_reset', 'query_id', 'tfidf_similarity',
       'keyword_lang', 'keyword_length', 'intent', 'product_ctr',
       'designer_ctr', 'productclass_ctr', 'class_ctr', 'lang_latin',
       'lang_mixed', 'lang_other', 'intent_navigational'],
      dtype='str')

In [17]:
# fill 28 productclass nulls
search_df_model["productclass_ctr"] = search_df_model.productclass_ctr.fillna(search_df_model.productclass_ctr.mean())

In [37]:
final_engineered_features = ["index"] + product_features + query_features + interaction_features + one_hot_features 

In [83]:
final_engineered_features = product_features +  interaction_features + one_hot_features  

In [69]:
# Train/test split
# Get unique query sessions 
# query signature cols hold the 3 composite keys that represent a session

from sklearn.model_selection import train_test_split

query_sessions = search_df_model[query_signature_final_cols].drop_duplicates()

# Split at query level
train_sessions, test_sessions = train_test_split(query_sessions, test_size=0.2, random_state=42)

# Map back to rows
train_df = search_df_model.merge(train_sessions, on=query_signature_final_cols, how="inner")
test_df = search_df_model.merge(test_sessions, on=query_signature_final_cols, how="inner")

print(f"Train: {len(train_df)} rows, {len(train_sessions)} queries")
print(f"Test: {len(test_df)} rows, {len(test_sessions)} queries")

Train: 3128638 rows, 204198 queries
Test: 781619 rows, 51050 queries


In [84]:
baseline_v1_train_model_data = train_df[final_engineered_features]
target_variables = train_df.plp_click.values

In [85]:
baseline_v1_test_model_data = test_df[final_engineered_features]
test_target_variables = test_df.plp_click.values

In [ ]:
baseline_v1_test_model_data["index"] = baseline_v1_train_model_data["index"].mean() # Neutralize testing index

In [86]:
from sklearn.preprocessing import StandardScaler

bv1_scaler = StandardScaler()
baseline_v1_train_model_data_scaled = bv1_scaler.fit_transform(baseline_v1_train_model_data)


In [87]:
baseline_v1_test_model_data_scaled = bv1_scaler.transform(baseline_v1_test_model_data)

## Baseline MRR performance of current results

In [ ]:
# All test query sessions, not just clicked ones
all_test_queries = test_df[query_signature_final_cols].drop_duplicates()

# For each query session, rank by original index order
test_df["original_rank"] = test_df.groupby(query_signature_final_cols)["index"].rank(ascending=True, method="first")

# For each query session, find the rank of the first clicked product
# If no click in session, this query contributes 0 (via fillna)
first_click_rank = (
    test_df[test_df["plp_click"] == 1]
    .groupby(query_signature_final_cols)["original_rank"]
    .min()
    .reset_index()
    .rename(columns={"original_rank": "first_click_rank"})
)

# Merge back to all query sessions, not just clicked ones
baseline0_eval = all_test_queries.merge(first_click_rank, on=query_signature_final_cols, how="left")
baseline0_eval["reciprocal_rank"] = (1 / baseline0_eval["first_click_rank"]).fillna(0)

mrr_baseline0 = baseline0_eval["reciprocal_rank"].mean()
print(f"Baseline 0 MRR: {mrr_baseline0:.4f}")

Baseline 0 MRR: 0.2462


## Feature Selection via Ablation

Rather than selecting features by intuition, a systematic ablation was run across all 63 non-empty subsets of the 6 feature groups. Each combination was evaluated using logistic regression (fast iteration) with position neutralized at test time.

**Key findings:**

| Rank | Features | n_features | MRR |
|------|----------|------------|-----|
| 1 | intent + kw_length | 3 | 0.2462 |
| 1 | lang + kw_length | 5 | 0.2462 |
| 1 | lang | 4 | 0.2462 |
| 1 | intent | 2 | 0.2462 |
| 1 | kw_length | 2 | 0.2462 |
| 1 | lang + intent | 5 | 0.2462 |
| 1 | lang + intent + kw_length | 6 | 0.2462 |
| 8 | interaction + lang + kw_length | 9 | 0.2346 |
| 9 | interaction (any combo) | 5–10 | 0.2345–0.2346 |
| — | tfidf + any combo | 5–10 | ≤ 0.2125 |

**Baseline 0 (current system): 0.2462**

Observations:

1. **Session-level features match but cannot beat the current system.** Intent, language, and keyword length all reach 0.2462 — exactly matching Baseline 0 — but none exceed it. These features are clean signals but they don't vary within a session (every impression in the same session shares the same keyword), so they cannot reorder products.

2. **Interaction features degrade ranking.** Binary substring matching is too crude to improve on the current ordering. Within a navigational session (e.g., "Gucci"), every Gucci product gets `kw_matches_designer=1`, providing no differentiation.

3. **TF-IDF similarity hurts.** Despite being the only continuous query-product relevance signal, TF-IDF cosine similarity between keyword and product name introduces noise that actively degrades ranking below the current system.

4. **CTR features hurt most** (not shown in top rows). Query-independent popularity features boost globally popular products regardless of query relevance, producing the lowest MRR scores in the ablation (≤ 0.1888).

5. **The structural ceiling:** The only features that could reorder products within a session are those that vary across products (CTR, interaction flags, TF-IDF). All of these degrade ranking because they introduce noise that overrides the current system's ordering. Features that are clean (intent, language) can't rerank because they're constant within a session. This is a feature information constraint, not a model capacity constraint — confirmed by convergence across logistic regression, LightGBM, and SVM.

In [53]:
"""
Feature Ablation Experiment
Run every combination of feature groups to find what helps and what hurts.
Uses logistic regression for speed.
"""

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import pandas as pd
import numpy as np
import time

# --- Feature groups ---
feature_groups = {
    "tfidf":       ["tfidf_similarity"],
    "interaction":  ["kw_matches_designer", "kw_matches_class", "kw_matches_productclass", "kw_in_product_name"],
    "ctr":          ["product_ctr", "designer_ctr", "productclass_ctr", "class_ctr"],
    "lang":         ["lang_latin", "lang_mixed", "lang_other"],
    "intent":       ["intent_navigational"],
    "kw_length":    ["keyword_length"],
}

# Position is always included (neutralized at inference)
position_col = ["index"]

# --- MRR computation function ---
def compute_mrr(test_df, prob_col, query_sig_cols):
    all_sessions = test_df[query_sig_cols].drop_duplicates()
    test_df["_pred_rank"] = test_df.groupby(query_sig_cols)[prob_col].rank(ascending=False, method="first")
    
    first_click = (
        test_df[test_df["plp_click"] == 1]
        .groupby(query_sig_cols)["_pred_rank"]
        .min()
        .reset_index()
        .rename(columns={"_pred_rank": "first_click_rank"})
    )
    
    eval_df = all_sessions.merge(first_click, on=query_sig_cols, how="left")
    eval_df["rr"] = (1 / eval_df["first_click_rank"]).fillna(0)
    return eval_df["rr"].mean()

# --- Run all non-empty subsets of feature groups ---
results = []
group_names = list(feature_groups.keys())

for r in range(1, len(group_names) + 1):
    for combo in combinations(group_names, r):
        # Assemble feature list
        feature_cols = position_col.copy()
        for g in combo:
            feature_cols.extend(feature_groups[g])
        
        # Prepare data
        X_train = train_df[feature_cols].copy()
        X_test = test_df[feature_cols].copy()
        X_test["index"] = X_train["index"].mean()  # neutralize position
        
        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train
        lr = LogisticRegression(class_weight="balanced", max_iter=1000, solver="lbfgs", C=1.0)
        t0 = time.time()
        lr.fit(X_train_scaled, train_df["plp_click"].values)
        train_time = time.time() - t0
        
        # Predict
        y_pred = lr.predict_proba(X_test_scaled)[:, 1]
        test_df["_click_prob"] = y_pred
        
        # Evaluate
        mrr = compute_mrr(test_df, "_click_prob", query_signature_final_cols)
        
        results.append({
            "features": " + ".join(combo),
            "n_features": len(feature_cols),
            "mrr": round(mrr, 4),
            "train_sec": round(train_time, 1)
        })
        
        print(f"  {' + '.join(combo):60s} → MRR: {mrr:.4f}  ({train_time:.1f}s)")

# Clean up temp columns
test_df.drop(columns=["_pred_rank", "_click_prob"], inplace=True, errors="ignore")

# --- Summary table ---
results_df = pd.DataFrame(results).sort_values("mrr", ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("FEATURE ABLATION RESULTS (sorted by MRR)")
print("="*80)
print(f"\nBaseline 0 (current system): 0.2462\n")
print(results_df.to_string(index=False))

  tfidf                                                        → MRR: 0.2091  (0.9s)
  interaction                                                  → MRR: 0.2345  (1.1s)
  ctr                                                          → MRR: 0.1881  (0.9s)
  lang                                                         → MRR: 0.2462  (0.7s)
  intent                                                       → MRR: 0.2462  (0.6s)
  kw_length                                                    → MRR: 0.2462  (0.7s)
  tfidf + interaction                                          → MRR: 0.2113  (0.8s)
  tfidf + ctr                                                  → MRR: 0.1885  (1.0s)
  tfidf + lang                                                 → MRR: 0.2118  (0.7s)
  tfidf + intent                                               → MRR: 0.2063  (0.7s)
  tfidf + kw_length                                            → MRR: 0.2028  (0.7s)
  interaction + ctr                                            → 

In [54]:
results_df[:20]

,features,n_features,mrr,train_sec
0,intent + kw_length,3,0.2462,1.0
1,lang + kw_length,5,0.2462,0.7
2,lang,4,0.2462,0.7
3,intent,2,0.2462,0.6
4,kw_length,2,0.2462,0.7
5,lang + intent,5,0.2462,0.8
6,lang + intent + kw_length,6,0.2462,0.8
7,interaction + lang + kw_length,9,0.2346,0.8
8,interaction + lang + intent + kw_length,10,0.2346,0.8
9,interaction + lang,8,0.2346,0.7


## Models experimentation (LR, LGMB)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight="balanced", max_iter=1000)

lr.fit(baseline_v1_train_model_data_scaled,target_variables)

y_pred = lr.predict_proba(baseline_v1_test_model_data_scaled)[:,1]

In [89]:
test_df["click_probability"] = y_pred

# All test query sessions, not just clicked ones
all_test_queries = test_df[query_signature_final_cols].drop_duplicates()

# For each query session, rank by original index order
test_df["predicted_rank"] = test_df.groupby(query_signature_final_cols)["click_probability"].rank(ascending=False, method="first")

# For each query session, find the rank of the first clicked product
# If no click in session, this query contributes 0 (via fillna)
model_first_click_rank = (
    test_df[test_df["plp_click"] == 1]
    .groupby(query_signature_final_cols)["predicted_rank"]
    .min()
    .reset_index()
    .rename(columns={"predicted_rank": "model_first_click_rank"})
)

# Merge back to all query sessions, not just clicked ones
baselinev1_eval = all_test_queries.merge(model_first_click_rank, on=query_signature_final_cols, how="left")
baselinev1_eval["reciprocal_rank"] = (1 / baselinev1_eval["model_first_click_rank"]).fillna(0)

mrr_baseline0 = baselinev1_eval["reciprocal_rank"].mean()
print(f"Baseline 1 MRR: {mrr_baseline0:.4f}")

Baseline 1 MRR: 0.1881


In [ ]:
import os
os.environ.pop("MPLBACKEND", None)

import matplotlib
matplotlib.use("agg")

import lightgbm as lgb

# Train
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=50,       # approximate ratio of negatives to positives
    min_child_samples=100,     # prevent overfitting on sparse click signal
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

model.fit(baseline_v1_train_model_data, target_variables)
y_pred = model.predict_proba(baseline_v1_test_model_data)[:, 1]


In [ ]:
test_df["click_probability"] = y_pred

# All test query sessions, not just clicked ones
all_test_queries = test_df[query_signature_final_cols].drop_duplicates()

# For each query session, rank by original index order
test_df["predicted_rank"] = test_df.groupby(query_signature_final_cols)["click_probability"].rank(ascending=False, method="first")

# For each query session, find the rank of the first clicked product
# If no click in session, this query contributes 0 (via fillna)
model_first_click_rank = (
    test_df[test_df["plp_click"] == 1]
    .groupby(query_signature_final_cols)["predicted_rank"]
    .min()
    .reset_index()
    .rename(columns={"predicted_rank": "model_first_click_rank"})
)

# Merge back to all query sessions, not just clicked ones
baselinev1_eval = all_test_queries.merge(model_first_click_rank, on=query_signature_final_cols, how="left")
baselinev1_eval["reciprocal_rank"] = (1 / baselinev1_eval["model_first_click_rank"]).fillna(0)

mrr_baseline0 = baselinev1_eval["reciprocal_rank"].mean()
print(f"Baseline 1 MRR: {mrr_baseline0:.4f}")

## Evaluation Method and Assumptions

### Primary metric: Mean Reciprocal Rank (MRR)

MRR measures how high the first clicked product appears in the model's re-ranked results, averaged across all test sessions. For each session, the reciprocal rank is 1/position of the first click (or 0 if no click occurred).

**Why MRR over nDCG:** 91% of successful sessions in this dataset have exactly one click. With a single relevant item per session, MRR and nDCG converge so nDCG's ability to credit multiple relevant items at different positions provides no additional signal.

**Zero-click sessions are included in the denominator.** Sessions with no clicks contribute 0 to MRR. This penalizes the metric for retrieval failures that no re-ranking model can fix, but it reflects total system performance. Excluding them would measure "how well does the system rank when it succeeds which is a less useful question.

### Supporting metrics (not computed, recommended for production)

**AUC** evaluates the model as a classifier independent of ranking. If AUC is poor (high bias/variance), any MRR improvement is accidental. If AUC is reasonable but MRR doesn't improve, the model learned click probability well but couldn't shift the ordering because of inherent data structure. That is diagnostic information about the feature set.

**Success@k** (fraction of sessions with at least one click in the top k results). It helps answer questions like "The model puts a clicked product in the top 5 for X% of sessions".  

### Assumptions

- **Click as relevance proxy.** The model treats clicks as positive labels. Clicks are noisy (users click out of curiosity, position bias inflates clicks at high positions, and some relevant products go unclicked). Conversion would be a stronger signal but is too sparse (~0.1% of impressions) to train on.

- **Position bias handling.** Position (`index`) is included as a training feature so the model can learn to separate position effect from product relevance. At inference, position is set to a constant (training mean) for all rows, neutralizing its contribution. This is simpler than inverse propensity weighting (IPW), which requires accurate propensity estimation and introduces its own assumptions.

- **Single-day data.** All impressions come from one day. CTR features computed on this data are noisy estimates of true popularity. In production, these would be computed over a historical window. This also means the model cannot capture temporal patterns (day-of-week, recency, trending products).

## Tradeoffs

### Feature design

**Query-independent vs query-dependent features.** Aggregate CTR features (product, designer, class level) capture global popularity but are blind to query relevance. The ablation confirmed this: CTR features produced the lowest MRR scores. Query-dependent features (TF-IDF similarity, substring matching) attempt to measure relevance but are too coarse in their current form. Substring matching is binary and misses morphological variants ("dresses" vs "dress"). TF-IDF operates in lexical  (as defined by the vocabulary) and fails on cross-lingual queries (Arabic keywords against English product names). The ideal feature (a continuous, cross-lingual relevance score per query-product pair) requires embedding-based retrieval, which is an approach to be investigated in Part 3.


### Model selection

**Logistic regression, LightGBM, and SVM were tested.** All three converged to the same MRR ceiling (0.2462), confirming that the bottleneck is feature information, not model capacity. Logistic regression was used for the ablation study due to fast iteration (~1 second per run). LightGBM was tested to check whether nonlinear feature interactions could break the ceiling. They did not.

**Simple model.** A logistic regression model with intent and index position as features attempt to best address structural data limitations (good signals are lacking, we do not need to optimize on complex models).

### Position bias correction

**Include-and-neutralize vs inverse propensity weighting (IPW).** Position is included as a training feature and set to a constant at inference. This is simpler than IPW, which requires estimating the probability that a user examines each position. IPW was also tested, using the observed click rate per position as a proxy for examination probability. Results were equivalent, confirming that the bottleneck is feature information, not the debiasing method.

### Ranking vs retrieval

**The central tradeoff of this case study.** The click model matches but cannot exceed the current system's MRR. This is because the current system already ranks reasonably well when it retrieves relevant products (among sessions with at least one click, the first click typically appears near position 1 (MRR of 0.86 on clicked sessions only)).

The performance bottleneck is the 68.7% of sessions that produce zero clicks. These are retrieval failures: the system did not surface any relevant product for the user to click. No re-ranking model can fix a session where the relevant product was never retrieved.
